In [1]:
import pandas as pd
import numpy as np

# Load cleaned data

In [2]:
# path data/processed/cleaned_citibike_data.parquet
df = pd.read_parquet("../data/processed/cleaned_citibike_data.parquet")
print(df.shape)
print(df.columns)

(5366163, 24)
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'end_is_dockless', 'start_is_dockless', 'start_hour',
       'start_dow', 'is_weekend', 'trip_duration', 'is_long_trip',
       'trip_distance_km', 'is_short_trip', 'start_id_nonstandard',
       'end_id_nonstandard'],
      dtype='object')


In [3]:
# Confirm station IDs loaded as float 
# and other types remain the same after using parquet format
df.dtypes

ride_id                         object
rideable_type                   object
started_at              datetime64[ns]
ended_at                datetime64[ns]
start_station_name      string[python]
start_station_id               float64
end_station_name        string[python]
end_station_id                 float64
start_lat                      float64
start_lng                      float64
end_lat                        float64
end_lng                        float64
member_casual                   object
end_is_dockless                   bool
start_is_dockless                 bool
start_hour                       int32
start_dow                        int32
is_weekend                        bool
trip_duration                  float64
is_long_trip                      bool
trip_distance_km               float64
is_short_trip                     bool
start_id_nonstandard              bool
end_id_nonstandard                bool
dtype: object

In [4]:
# visual check of the first few rows
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,start_is_dockless,start_hour,start_dow,is_weekend,trip_duration,is_long_trip,trip_distance_km,is_short_trip,start_id_nonstandard,end_id_nonstandard
0,E8950F16E8963131,electric_bike,2026-06-18 08:54:16.609,2026-06-18 09:06:26.757,E 6 St & Ave D,5506.14,E 51 St & 2 Ave,6575.03,40.722281,-73.976687,...,False,8,3,False,730.148,False,3.749067,False,False,False
1,AB5E676A98A98FFD,electric_bike,2026-06-18 18:10:49.259,2026-06-18 18:13:07.198,Broadway & Roebling St,5125.07,Broadway & Berry St,5164.05,40.709248,-73.960631,...,False,18,3,False,137.939,False,0.412818,False,False,False
2,7ACF812D3851E83B,electric_bike,2026-06-30 18:11:01.624,2026-06-30 18:27:34.826,Columbus Ave & W 59 St,6986.07,W 25 St & 9 Ave,6339.06,40.769310,-73.984640,...,False,18,1,False,993.202,False,2.739368,False,False,False
3,66955C7329D73810,electric_bike,2026-06-22 07:37:19.887,2026-06-22 07:52:03.386,E 14 St & Ave B,5736.09,Washington St & Laight St,5509.02,40.729387,-73.977724,...,False,7,0,False,883.499,False,2.879961,False,False,False
4,1020978B6F86D90F,electric_bike,2026-06-30 07:45:23.969,2026-06-30 07:59:28.521,E 14 St & Ave B,5736.09,Washington St & Laight St,5509.02,40.729387,-73.977724,...,False,7,1,False,844.552,False,2.879961,False,False,False


# Deciding station identity strategy

Using start station as the primary unit since that's where the rider decides to take the trip, which is most relevant to the conversion question.

In [5]:
# confirm that station IDs are unique to a single name and coordinates
station_check = df.groupby('start_station_id').agg(
    n_names=('start_station_name', 'nunique'),
    n_lats=('start_lat', 'nunique'),
    n_lngs=('start_lng', 'nunique'),
)
inconsistent = station_check[(station_check['n_names'] > 1) |
                              (station_check['n_lats'] > 1) |
                              (station_check['n_lngs'] > 1)]
print(f"Stations with inconsistent name/coords: {len(inconsistent)}")

Stations with inconsistent name/coords: 4


In [6]:
# show specific station IDs with inconsistent name/coords
print(inconsistent)

                  n_names  n_lats  n_lngs
start_station_id                         
5303.06                 1       2       2
5308.04                 2       1       1
6517.08                 1       2       2
6569.09                 1       2       2


In [7]:
# haversine function + loop for stations with multiple coordinates
# measures how far apart two points are on the surface of a sphere, 
# given their lat/lon coordinates

from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

for sid in [5303.06, 6517.08, 6569.09]:
    coords = df[df['start_station_id'] == sid][['start_lat', 'start_lng']].drop_duplicates().values
    if len(coords) == 2:
        dist = haversine_km(coords[0][0], coords[0][1], coords[1][0], coords[1][1])
        print(f"{sid}: distance between coordinate pairs = {dist*1000:.1f} meters")

5303.06: distance between coordinate pairs = 16.2 meters
6517.08: distance between coordinate pairs = 12.3 meters
6569.09: distance between coordinate pairs = 32.9 meters


A few station IDs had minor GPS coordinate variance (under ~35m) across trips at the same physical dock, that can be assumed as noise rather than distinct locations.

# Build station-level trip counts

In [8]:
# count the number of trips starting at each station
total_trips = df.groupby('start_station_id').size().rename('total_trips')
total_trips.head()

start_station_id
1964.01    129
2009.04    221
2042.01     87
2086.07    121
2118.13     95
Name: total_trips, dtype: int64

In [9]:
#count casual vs member trips starting at each station

casual_trips = (
    df[df['member_casual'] == 'casual']
    .groupby('start_station_id')
    .size()
    .rename('casual_trips')
)
member_trips = (
    df[df['member_casual'] == 'member']
    .groupby('start_station_id')
    .size()
    .rename('member_trips')
)

print(casual_trips.head())
print(member_trips.head())

start_station_id
1964.01     74
2009.04    124
2042.01     37
2086.07     53
2118.13     35
Name: casual_trips, dtype: int64
start_station_id
1964.01    55
2009.04    97
2042.01    50
2086.07    68
2118.13    60
Name: member_trips, dtype: int64


In [10]:
# combine the three series into a single dataframe

station_agg = pd.concat([total_trips, casual_trips, member_trips], axis=1)
station_agg = station_agg.fillna(0)  # stations with 0 casual or 0 member trips
station_agg.head()

,total_trips,casual_trips,member_trips
start_station_id,,,
1964.01,129,74,55
2009.04,221,124,97
2042.01,87,37,50
2086.07,121,53,68
2118.13,95,35,60


# Compute casual-ride share per station

In [11]:
# compute the share of casual trips for each station
station_agg['casual_share'] = station_agg['casual_trips'] / station_agg['total_trips']
station_agg[['total_trips', 'casual_trips', 'member_trips', 'casual_share']].describe()

,total_trips,casual_trips,member_trips,casual_share
count,2271.00000,2271.000000,2271.000000,2271.000000
mean,2362.90753,483.870101,1879.037428,0.229241
std,2862.94235,633.751039,2294.144058,0.080594
min,2.00000,1.000000,1.000000,0.070784
25%,375.00000,89.000000,291.000000,0.174675
50%,1065.00000,221.000000,850.000000,0.210733
75%,3504.50000,689.500000,2792.500000,0.263470
max,18076.00000,7694.000000,15684.000000,0.724171


# Add behavioral features per station

In [12]:
# average trip duration by station and user type
duration_by_type = (
    df.groupby(['start_station_id', 'member_casual'])['trip_duration']
    .mean()
    .unstack()
    .rename(columns={'casual': 'avg_duration_casual', 'member': 'avg_duration_member'})
)
duration_by_type.head()

member_casual,avg_duration_casual,avg_duration_member
start_station_id,,
1964.01,1844.414838,831.917691
2009.04,2190.059194,1280.789794
2042.01,1341.828730,885.973300
2086.07,1287.880925,972.359103
2118.13,998.027571,1041.836950


In [13]:
# average trip distance by station and user type
distance_by_type = (
    df.groupby(['start_station_id', 'member_casual'])['trip_distance_km']
    .mean()
    .unstack()
    .rename(columns={'casual': 'avg_distance_casual', 'member': 'avg_distance_member'})
)
distance_by_type.head()

member_casual,avg_distance_casual,avg_distance_member
start_station_id,,
1964.01,1.602987,2.110017
2009.04,1.260496,2.356308
2042.01,1.852073,1.980746
2086.07,1.994216,2.498095
2118.13,1.761628,2.570897


In [14]:
# share of trips starting on weekends by station
weekend_share = (
    df.groupby('start_station_id')['is_weekend']
    .mean()
    .rename('weekend_trip_share')
)

print(weekend_share.head())

start_station_id
1964.01    0.248062
2009.04    0.438914
2042.01    0.310345
2086.07    0.297521
2118.13    0.252632
Name: weekend_trip_share, dtype: float64


In [15]:
# most common start hour for each station
peak_hour = (
    df.groupby('start_station_id')['start_hour']
    .agg(lambda x: x.mode()[0])
    .rename('peak_hour')
)

In [16]:
# combine all behavioral metrics into the main station_agg dataframe
station_agg = station_agg.join([duration_by_type, distance_by_type, weekend_share, peak_hour])
station_agg.head()

,total_trips,casual_trips,member_trips,casual_share,avg_duration_casual,avg_duration_member,avg_distance_casual,avg_distance_member,weekend_trip_share,peak_hour
start_station_id,,,,,,,,,,
1964.01,129,74,55,0.573643,1844.414838,831.917691,1.602987,2.110017,0.248062,18
2009.04,221,124,97,0.561086,2190.059194,1280.789794,1.260496,2.356308,0.438914,20
2042.01,87,37,50,0.425287,1341.828730,885.973300,1.852073,1.980746,0.310345,13
2086.07,121,53,68,0.438017,1287.880925,972.359103,1.994216,2.498095,0.297521,13
2118.13,95,35,60,0.368421,998.027571,1041.836950,1.761628,2.570897,0.252632,17


# Bring in coordinates

In [17]:
# grab one representative coordinate for each station for mapping purposes
coords = df.groupby('start_station_id')[['start_lat', 'start_lng']].first()
coords.head()

,start_lat,start_lng
start_station_id,,
1964.01,40.61124,-74.03282
2009.04,40.61150,-74.03513
2042.01,40.61327,-74.03315
2086.07,40.61451,-74.03502
2118.13,40.61466,-74.02946


In [18]:
# attach coordinates to the station_agg dataframe
station_agg = station_agg.join(coords)
station_agg.head()

,total_trips,casual_trips,member_trips,casual_share,avg_duration_casual,avg_duration_member,avg_distance_casual,avg_distance_member,weekend_trip_share,peak_hour,start_lat,start_lng
start_station_id,,,,,,,,,,,,
1964.01,129,74,55,0.573643,1844.414838,831.917691,1.602987,2.110017,0.248062,18,40.61124,-74.03282
2009.04,221,124,97,0.561086,2190.059194,1280.789794,1.260496,2.356308,0.438914,20,40.61150,-74.03513
2042.01,87,37,50,0.425287,1341.828730,885.973300,1.852073,1.980746,0.310345,13,40.61327,-74.03315
2086.07,121,53,68,0.438017,1287.880925,972.359103,1.994216,2.498095,0.297521,13,40.61451,-74.03502
2118.13,95,35,60,0.368421,998.027571,1041.836950,1.761628,2.570897,0.252632,17,40.61466,-74.02946


In [19]:
# bring in station name too, for readability
names = df.groupby('start_station_id')['start_station_name'].first()
station_agg = station_agg.join(names)

station_agg.head()

,total_trips,casual_trips,member_trips,casual_share,avg_duration_casual,avg_duration_member,avg_distance_casual,avg_distance_member,weekend_trip_share,peak_hour,start_lat,start_lng,start_station_name
start_station_id,,,,,,,,,,,,,
1964.01,129,74,55,0.573643,1844.414838,831.917691,1.602987,2.110017,0.248062,18,40.61124,-74.03282,101 St & Fort Hamilton Pkwy
2009.04,221,124,97,0.561086,2190.059194,1280.789794,1.260496,2.356308,0.438914,20,40.61150,-74.03513,Shore Rd & 4 Ave
2042.01,87,37,50,0.425287,1341.828730,885.973300,1.852073,1.980746,0.310345,13,40.61327,-74.03315,4 Ave & 99 St
2086.07,121,53,68,0.438017,1287.880925,972.359103,1.994216,2.498095,0.297521,13,40.61451,-74.03502,99 St & 3 Ave
2118.13,95,35,60,0.368421,998.027571,1041.836950,1.761628,2.570897,0.252632,17,40.61466,-74.02946,Fort Hamilton Pkwy & 95 St


# Filter unstable stations

In [20]:
#check the distribution of total trips across stations
station_agg['total_trips'].describe()

count     2271.00000
mean      2362.90753
std       2862.94235
min          2.00000
25%        375.00000
50%       1065.00000
75%       3504.50000
max      18076.00000
Name: total_trips, dtype: float64

In [21]:
# see how many stations have fewer than a certain number of trips for filtering decision purposes
for threshold in [10, 25, 50, 100, 200, 374]:
    n_below = (station_agg['total_trips'] < threshold).sum()
    print(f"Stations below {threshold} trips: {n_below} ({n_below/len(station_agg)*100:.1f}%)")

Stations below 10 trips: 4 (0.2%)
Stations below 25 trips: 6 (0.3%)
Stations below 50 trips: 12 (0.5%)
Stations below 100 trips: 53 (2.3%)
Stations below 200 trips: 226 (10.0%)
Stations below 374 trips: 564 (24.8%)


In [22]:
# set the filtering threshold for stations with very few trips (5th percentile)
cutoff = station_agg['total_trips'].quantile(0.05)  # bottom 5%
print(f"5th percentile cutoff: {cutoff:.0f} trips")

5th percentile cutoff: 144 trips


In [23]:
# drop stations with fewer than the cutoff number of trips
station_agg_filtered = station_agg[station_agg['total_trips'] >= cutoff].copy()
print(f"Shape before filtering: {station_agg.shape}")
print(f"Shape after filtering: {station_agg_filtered.shape}")

Shape before filtering: (2271, 13)
Shape after filtering: (2160, 13)


# Put everything together and save

In [24]:
# save final station-level summary for use in later notebooks 
# (as parquet for smaller file size and faster read/write)
output_path = "../data/processed/station_summary.parquet"
station_agg_filtered.reset_index().to_parquet(output_path, index=False)
print(f"Station summary saved to {output_path}")
print(station_agg_filtered.shape)

Station summary saved to ../data/processed/station_summary.parquet
(2160, 13)


In [25]:
# quick check of the saved output
station_agg_filtered.head()

,total_trips,casual_trips,member_trips,casual_share,avg_duration_casual,avg_duration_member,avg_distance_casual,avg_distance_member,weekend_trip_share,peak_hour,start_lat,start_lng,start_station_name
start_station_id,,,,,,,,,,,,,
2009.04,221,124,97,0.561086,2190.059194,1280.789794,1.260496,2.356308,0.438914,20,40.61150,-74.03513,Shore Rd & 4 Ave
2158.08,197,56,141,0.284264,1188.031786,831.328362,1.315942,2.146326,0.248731,19,40.61689,-74.03091,4 Ave & 94 St
2190.08,195,71,124,0.364103,1471.565577,1297.978427,2.855367,4.115211,0.246154,14,40.61717,-74.02744,Fort Hamilton Pkwy & 92 St
2231.10,155,68,87,0.438710,1098.562206,899.890874,2.046109,2.715366,0.374194,7,40.61946,-74.03261,3 Ave & 92 St
2240.06,170,83,87,0.488235,1366.095120,914.267575,1.550043,2.259779,0.305882,19,40.61919,-74.04020,Shore Rd & Oliver St
